In [8]:
import numpy as np
import json

X_train = np.load('../data/X_train.npy', allow_pickle=True)
X_val   = np.load('../data/X_val.npy', allow_pickle=True)
X_test  = np.load('../data/X_test.npy', allow_pickle=True)
y_train = np.load('../data/y_train.npy', allow_pickle=True)
y_val   = np.load('../data/y_val.npy', allow_pickle=True)
y_test  = np.load('../data/y_test.npy', allow_pickle=True)

with open('../data/feature_names.json') as f:
    feature_names = json.load(f)

print(X_train.shape, X_val.shape, X_test.shape)

(4929, 41) (1057, 41) (1057, 41)


In [9]:
X_train = X_train.astype('float32')
X_val   = X_val.astype('float32')
X_test  = X_test.astype('float32')
y_train = y_train.astype('float32')
y_val   = y_val.astype('float32')
y_test  = y_test.astype('float32')

print(X_train.dtype)   # should now show float32

float32


In [10]:
import tensorflow as tf, keras

In [11]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)


In [12]:
from keras.src.callbacks import early_stopping
from tensorflow.keras import layers, regularizers

model_v2 = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.3),
    layers.Dense(16, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model_v2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history_v2 = model_v2.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50, batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/50


155/155 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7249 - loss: 0.5882 - val_accuracy: 0.7815 - val_loss: 0.5009
Epoch 2/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7809 - loss: 0.4972 - val_accuracy: 0.7938 - val_loss: 0.4747
Epoch 3/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7837 - loss: 0.4774 - val_accuracy: 0.7966 - val_loss: 0.4662
Epoch 4/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7996 - loss: 0.4687 - val_accuracy: 0.7909 - val_loss: 0.4578
Epoch 5/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8042 - loss: 0.4547 - val_accuracy: 0.7975 - val_loss: 0.4571
Epoch 6/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8040 - loss: 0.4468 - val_accuracy: 0.7975 - val_loss: 0.4504
Epoch 7/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8065 - loss: 0.4445 - val_accuracy: 0.7928 - val_loss: 0.4487
Epoch 8/50
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8046 - loss: 0.4406 - val_accuracy: 0.7938 - val_

In [13]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred_proba_v2 = model_v2.predict(X_val).ravel()
y_pred_v2 = (y_pred_proba_v2 > 0.5).astype(int)

print(classification_report(y_val, y_pred_v2))
print(f"ROC-AUC: {roc_auc_score(y_val, y_pred_proba_v2):.4f}")

34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
              precision    recall  f1-score   support

         0.0       0.85      0.90      0.87       776
         1.0       0.67      0.56      0.61       281

    accuracy                           0.81      1057
   macro avg       0.76      0.73      0.74      1057
weighted avg       0.80      0.81      0.80      1057

ROC-AUC: 0.8469


In [14]:
model_v2.save('../models/neural_network_v2.keras')

In [15]:
y_pred_v2_035 = (y_pred_proba_v2 > 0.35).astype(int)
print(classification_report(y_val, y_pred_v2_035))

              precision    recall  f1-score   support

         0.0       0.88      0.79      0.83       776
         1.0       0.55      0.70      0.62       281

    accuracy                           0.77      1057
   macro avg       0.72      0.75      0.73      1057
weighted avg       0.79      0.77      0.78      1057



In [16]:
with open('../models/final_threshold.txt', 'w') as f:
    f.write('0.35')

## Phase 6: Final Model Selection — Regularized NN @ 0.35 Threshold

### Configurations Tested

| Config                     | Accuracy | Churn Precision | Churn Recall | Churn F1 | ROC-AUC |
|-----------------------------|:--------:|:----------------:|:-------------:|:---------:|:--------:|
| Logistic Regression (0.5)   | 0.80     | 0.64             | 0.55          | 0.59      | 0.8435   |
| Original NN (0.5)           | 0.80     | 0.65             | 0.53          | 0.58      | 0.8441   |
| Original NN (0.35)          | 0.78     | 0.58             | 0.69          | 0.63      | 0.8441   |
| Regularized NN v2 (0.5)     | 0.81     | 0.67             | 0.56          | 0.61      | 0.8469   |
| **Regularized NN v2 (0.35)**| **0.77** | **0.55**         | **0.70**      | 0.62      | **0.8469** |

### Regularization Impact

Adding Dropout (0.3) and L2 regularization (0.001) to the original architecture 
changed the training dynamics noticeably: validation loss continued improving 
through at least epoch 30, compared to the original model plateauing and 
triggering `EarlyStopping` at epoch 10. This is direct evidence the network 
was previously overfitting, and regularization meaningfully extended useful 
training. Every evaluation metric improved as a result, including ROC-AUC 
(0.8441 → 0.8469) — now clearly ahead of Logistic Regression (0.8435), rather 
than statistically tied.

### Final Decision: Regularized NN v2 @ 0.35 Threshold

I selected this configuration as the final model because it achieves the 
**highest churn recall (0.70)** of every configuration tested — built on the 
better-performing regularized model rather than the original.

This decision follows directly from the business framing established in 
Phase 1: for a retention use case, failing to identify an actual churner 
(a lost customer, and their lifetime value) is a costlier mistake than 
flagging a loyal customer who was never going to leave (a bounded cost — 
a retention offer or outreach call that wasn't strictly necessary).

**Trade-off accepted:** precision drops to 0.55, meaning close to half of 
customers flagged as "at risk" will not actually churn. This is a deliberate 
choice, not an oversight — in a real deployment, this threshold would ideally 
be revisited with actual